# BLEU
"BLEU: a Method for Automatic Evaluation of Machine Translation" (Papineni et al., 2002)

**Idea:** score a machine translation by how many of its **n-grams** (word sequences of length 1 to 4) also appear in human reference translations, and punish translations that are too short. No model, no training: just counting.

### BLEU vs other text metrics
| | Exact match | BLEU | ROUGE | Perplexity | Learned (BERTScore, COMET) |
|---|---|---|---|---|---|
| Compares | whole string | n-grams, 1 to 4 | n-grams / longest common subsequence | model probability of the text | embeddings from a pretrained encoder |
| Oriented to | - | **precision** (is what you said in the reference?) | **recall** (did you cover the reference?) | - | both |
| Length control | - | brevity penalty | - | per-token average | - |
| Needs references | yes | yes (1 or more) | yes | no | yes |
| Rewards paraphrase | no | no | no | - | yes |
| Used for | QA, classification (`bert.ipynb`) | translation | summarization | language models (`slm.ipynb`) | translation, generation |

Here: every piece from scratch in plain Python, checked on the worked examples from the paper, then the failure cases that make BLEU a rough tool.

Flow: `candidate + references (word lists) -> clipped n-gram counts for n = 1..4 -> precisions p_1..p_4 -> geometric mean x brevity penalty -> BLEU in [0, 1]`

In [ ]:
import math
from collections import Counter

import matplotlib.pyplot as plt

## 1. n-grams
An n-gram is a run of $n$ consecutive tokens. A sentence of $L$ tokens has $L - n + 1$ of them.

Tokenization here is lowercase + split on spaces. (The choice matters a lot in practice, see the notes.)

In [ ]:
def tokenize(text):
    return text.lower().split()

def ngrams(tokens, n):
    # list of L tokens -> Counter over the L - n + 1 n-grams (as tuples)
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

s = tokenize("the cat is on the mat")
print(ngrams(s, 1))
print(ngrams(s, 2))

## 2. Modified (clipped) n-gram precision
Plain precision = "what fraction of the candidate's words appear in a reference". It is trivially gamed: the candidate `the the the the the the the` has precision 7/7 against `the cat is on the mat`.

The fix: each candidate n-gram is credited at most as many times as it occurs in any **single** reference.

### Formula: Modified n-gram precision
$$p_n = \frac{\sum_{g \in \text{n-grams}(c)} \min\big(\text{Count}_c(g),\ \max_{r \in R} \text{Count}_r(g)\big)}{\sum_{g \in \text{n-grams}(c)} \text{Count}_c(g)}$$
- $c$: candidate translation; $R$: its set of reference translations
- $\text{Count}_c(g)$: times n-gram $g$ occurs in the candidate
- $\max_{r} \text{Count}_r(g)$: the most times $g$ occurs in any one reference (the clip)
- numerator: clipped matches; denominator: number of n-grams in the candidate, $|c| - n + 1$

In [ ]:
def clipped_counts(cand, refs, n):
    # cand: list of tokens, refs: list of token lists -> (clipped matches, total candidate n-grams)
    c = ngrams(cand, n)
    max_ref = Counter()
    for r in refs:
        for g, k in ngrams(r, n).items():
            max_ref[g] = max(max_ref[g], k)
    matches = sum(min(k, max_ref[g]) for g, k in c.items())
    return matches, sum(c.values())

# example 2 from the paper
cand = tokenize("the the the the the the the")
refs = [tokenize("the cat is on the mat"), tokenize("there is a cat on the mat")]
print("plain precision   :", sum(w in refs[0] for w in cand), "/", len(cand))
print("modified precision:", "%d / %d" % clipped_counts(cand, refs, 1))   # 'the' is clipped at 2

### The paper's example 1
Two candidate translations of the same Chinese sentence, three references. Candidate 1 is clearly better; the paper reports $p_1 = 17/18$, $p_2 = 10/17$ for it and $p_1 = 8/14$, $p_2 = 1/13$ for candidate 2.

In [ ]:
cand1 = tokenize("It is a guide to action which ensures that the military always obeys the commands of the party")
cand2 = tokenize("It is to insure the troops forever hearing the activity guidebook that party direct")
refs = [tokenize(r) for r in [
    "It is a guide to action that ensures that the military will forever heed Party commands",
    "It is the guiding principle which guarantees the military forces always being under the command of the Party",
    "It is the practical guide for the army always to heed the directions of the party",
]]
for name, c in [("candidate 1", cand1), ("candidate 2", cand2)]:
    print(name, "| " + " | ".join("p_%d = %d/%d" % (n, *clipped_counts(c, refs, n)) for n in range(1, 5)))

assert clipped_counts(cand1, refs, 1) == (17, 18) and clipped_counts(cand1, refs, 2) == (10, 17)
assert clipped_counts(cand2, refs, 1) == (8, 14) and clipped_counts(cand2, refs, 2) == (1, 13)

## 3. Brevity penalty
Precision alone rewards saying very little: the candidate `of the` scores $p_1 = 2/2$ and $p_2 = 1/1$ against the references above. Recall would fix this, but recall is awkward with several references (a good translation matches *one* of them, not all).

BLEU instead multiplies by a penalty when the candidate is shorter than the reference.

### Formula: Brevity penalty
$$\text{BP} = \begin{cases} 1 & c > r \\ e^{\,1 - r / c} & c \le r \end{cases}$$
- $c$: total candidate length (tokens, summed over the corpus)
- $r$: **effective reference length**: for each sentence, the reference length closest to the candidate's length (ties -> the shorter), summed over the corpus
- no bonus for being longer: extra words already hurt precision

In [ ]:
def closest_ref_len(cand, refs):
    return min((abs(len(r) - len(cand)), len(r)) for r in refs)[1]   # closest length, ties -> shorter

def brevity_penalty(c, r):
    if c == 0:
        return 0.0
    return 1.0 if c > r else math.exp(1 - r / c)

short = tokenize("of the")
print("'of the': p_1 = %d/%d, p_2 = %d/%d" % (*clipped_counts(short, refs, 1), *clipped_counts(short, refs, 2)),
      "| BP =", round(brevity_penalty(len(short), closest_ref_len(short, refs)), 5))

ratio = [i / 100 for i in range(20, 151)]
plt.figure(figsize=(5, 3))
plt.plot(ratio, [brevity_penalty(x, 1.0) for x in ratio])
plt.xlabel("candidate length / reference length"); plt.ylabel("BP"); plt.show()

## 4. BLEU
Combine the four precisions with a **geometric** mean (so a zero in any one drives the score to zero, and the fast-decaying higher-order precisions still count), then apply the brevity penalty.

BLEU is a **corpus-level** metric: matches, totals and lengths are summed over all sentences first, and the ratio is taken once. It is *not* the average of per-sentence scores.

### Formula: BLEU
$$\text{BLEU} = \text{BP} \cdot \exp\!\left(\sum_{n=1}^{N} w_n \log p_n\right), \qquad p_n = \frac{\sum_{s} \text{matches}_n(s)}{\sum_{s} \text{total}_n(s)}$$
- $N = 4$, $w_n = 1/4$: the standard "BLEU-4"
- $s$: runs over the sentences of the test corpus
- $p_1$ measures **adequacy** (right words), $p_2..p_4$ measure **fluency** (right word order)
- range $[0, 1]$, usually reported $\times 100$; 1 only if the candidate equals a reference

In [ ]:
def bleu(cands, refs_list, max_n=4, smooth=False):
    # cands: list of token lists; refs_list: list (one per candidate) of lists of token lists
    # -> BLEU, [p_1..p_N], BP
    matches, totals = [0] * max_n, [0] * max_n
    c_len = r_len = 0
    for cand, refs in zip(cands, refs_list):
        c_len += len(cand)
        r_len += closest_ref_len(cand, refs)
        for n in range(1, max_n + 1):
            m, t = clipped_counts(cand, refs, n)
            matches[n - 1] += m
            totals[n - 1] += t
    if smooth:   # add-one on n >= 2 (Lin & Och, 2004), for single sentences
        precisions = [(m + (n > 0)) / (t + (n > 0)) if t + (n > 0) > 0 else 0.0 for n, (m, t) in enumerate(zip(matches, totals))]
    else:
        precisions = [m / t if t > 0 else 0.0 for m, t in zip(matches, totals)]
    bp = brevity_penalty(c_len, r_len)
    if min(precisions) == 0:
        return 0.0, precisions, bp
    return bp * math.exp(sum(math.log(p) for p in precisions) / max_n), precisions, bp

for name, c in [("candidate 1", cand1), ("candidate 2", cand2)]:
    score, ps, bp = bleu([c], [refs])
    print(f"{name}: BLEU {100 * score:5.2f} | precisions {[round(p, 3) for p in ps]} | BP {bp:.3f}")

## 5. Sentence-level BLEU needs smoothing
On a single sentence it is common to have **no** matching 4-gram, which makes $p_4 = 0$ and the whole score 0 no matter how good the rest is (candidate 2 above). Over a corpus this does not happen, which is why the paper only defines corpus BLEU.

When a per-sentence score is needed, the counts are smoothed.

### Formula: Add-one smoothing (Lin & Och, 2004)
$$p_n = \frac{\text{matches}_n + 1}{\text{total}_n + 1} \quad (n \ge 2)$$
- only for $n \ge 2$; $p_1$ stays exact, so a sentence with no word in common still scores 0
- one of several schemes (Chen & Cherry, 2014 compare seven); scores from different schemes are not comparable

In [ ]:
pairs = [
    ("the cat sat on the mat", "the cat sat on the mat"),
    ("the cat sat on the mat", "the cat is sitting on the mat"),
    ("the cat sat on the mat", "a cat sat on a mat"),
    ("the cat sat on the mat", "on the mat sat the cat"),
    ("the cat sat on the mat", "the the the the the the"),
]
print(f"{'candidate':32s} {'BLEU':>6s} {'smoothed':>9s}   precisions")
for ref, cand in pairs:
    raw, ps, _ = bleu([tokenize(cand)], [[tokenize(ref)]])
    sm, _, _ = bleu([tokenize(cand)], [[tokenize(ref)]], smooth=True)
    print(f"{cand:32s} {100 * raw:6.2f} {100 * sm:9.2f}   {[round(p, 2) for p in ps]}")

## 6. Corpus BLEU is not the mean of sentence BLEUs
Summing counts first lets long sentences contribute more and lets a missing 4-gram in one sentence be covered by the others. A small "test set" of four sentence pairs:

In [ ]:
corpus = [
    ("the quick brown fox jumps over the lazy dog", "the quick brown fox jumped over the lazy dog"),
    ("i would like a cup of coffee please", "i want a cup of coffee please"),
    ("she has been living in paris for ten years", "she lived in paris for ten years"),
    ("he is not coming home tonight", "he will not come home tonight"),   # no matching 4-gram -> sentence BLEU 0
]
cands = [tokenize(c) for _, c in corpus]
refs_list = [[tokenize(r)] for r, _ in corpus]

per_sentence = [bleu([c], [r])[0] for c, r in zip(cands, refs_list)]
score, ps, bp = bleu(cands, refs_list)
print("sentence BLEUs      :", [round(100 * s, 2) for s in per_sentence])
print(f"mean sentence BLEU  : {100 * sum(per_sentence) / len(per_sentence):.2f}")
print(f"corpus BLEU         : {100 * score:.2f} | precisions {[round(p, 3) for p in ps]} | BP {bp:.3f}")

## 7. More references -> higher BLEU
A translation can be right in many ways. Each extra reference adds n-grams that count as correct, so the score can only go up. Scores computed with different numbers of references are not comparable.

In [ ]:
for k in [1, 2, 3]:
    print(f"{k} reference(s): candidate 1 BLEU {100 * bleu([cand1], [refs[:k]])[0]:5.2f} | smoothed candidate 2 BLEU {100 * bleu([cand2], [refs[:k]], smooth=True)[0]:5.2f}")

## 8. Where BLEU fails
BLEU only sees surface strings. Against the reference `the film was very good`:
- a correct **paraphrase** with different words scores low
- a sentence with the **opposite meaning** that shares most words scores about twice as high as the paraphrase
- tokenization changes the score (`good.` vs `good .`), with no change in the translation

In [ ]:
ref = "the film was very good ."
for cand in [
    "the film was very good .",
    "the movie was really great .",      # same meaning, different words
    "the film was not very good .",      # opposite meaning, one extra word
    "the film was very good.",           # same sentence, punctuation not split off
]:
    sm, ps, bp = bleu([tokenize(cand)], [[tokenize(ref)]], smooth=True)
    print(f"{cand:32s} smoothed BLEU {100 * sm:6.2f} | precisions {[round(p, 2) for p in ps]} | BP {bp:.2f}")

## Notes
- BLEU correlates with human judgement at the **system** level (which of two MT systems is better over thousands of sentences), and poorly at the sentence level. It was designed for the first use only.
- It cannot be compared across papers unless tokenization, casing, number of references and smoothing all match. **sacreBLEU** (Post, 2018) exists to pin these down: it applies its own standard tokenizer to detokenized output and reports a signature string with the score.
- The original Transformer was evaluated with it: 28.4 BLEU on WMT14 English-German, 41.8 on English-French (`transformer.ipynb`).
- Weaknesses in order of practical impact: no credit for synonyms or paraphrase, no notion of meaning (negation barely changes the score), all words weigh the same (`not` = `the`), and it saturates for strong systems.
- Related metrics: **ROUGE** (recall-oriented, for summarization), **METEOR** (stems + synonyms + alignment), **chrF** (character n-gram F-score, better for morphologically rich languages), and learned metrics (**BERTScore**, **COMET**, **BLEURT**) that compare embeddings instead of strings and track human judgement much better.
- For LLM outputs BLEU is mostly replaced by task-specific checks and model-graded evals, but it remains the fast, free, reproducible baseline for translation.